# Solutions · 2. Data reconciliation and the detection of gross errors

Worked solutions to the three exercises of [Lecture 2](../../notebooks/part1_conservation/02_process_data_reconciliation_and_gross_error_detection.ipynb). Each solution is computed here and checked with `assert` statements against an exact identity or an independent calculation. The first code cell sets up the notebook and repeats the demonstrations of the lecture without printing them, because the solutions use their variables.

In [1]:
try:                      # use the installed package, or the copy in this repository
    import engml_2_applied
except ImportError:
    import pathlib, subprocess, sys
    for _root in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (_root / "engml_2_applied" / "__init__.py").exists():
            sys.path.insert(0, str(_root))
            break
    else:                 # on Google Colab: install from GitHub
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "git+https://github.com/ktwyw/engml-2-applied.git"], check=True)
    import engml_2_applied

import inspect
import numpy as np
import matplotlib.pyplot as plt
from engml_2_applied import (clustering_phases, constants, descriptors, dim_reduction, linear_physics,
                             trees_ensembles, uncertainty)
from engml_2_applied.plotting import panel_label, use_style

use_style()

def show_source(obj):
    """Return the source code of a library function or class."""
    return inspect.getsource(obj)

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

import io, sys
_stdout, sys.stdout = sys.stdout, io.StringIO()   # the demonstrations are repeated without their output
heading("Demonstration 1: 3-Stream Splitter Pen-and-Paper Micro-Example")
# Splitter node: Feed F1 splits into Product F2 and Recycle F3 -> F1 - F2 - F3 = 0
A_split = np.array([[1.0, -1.0, -1.0]])
b_split = np.array([0.0])
y_meas_split = np.array([102.0, 60.0, 39.0])  # Raw imbalance r = 102 - 60 - 39 = +3.0 kg/h
sigma_split = np.array([1.0, 1.0, 2.0])       # Variances = [1, 1, 4], sum = 6.0

rec_split = linear_physics.crowe_data_reconciliation(y_meas_split, sigma_split, A_split, b_split)
# Exact hand calculation: adjustments proportional to variance [-1/6 * 3, +1/6 * 3, +4/6 * 3] = [-0.5, +0.5, +2.0]
assert np.allclose(rec_split["y_reconciled"], [101.5, 60.5, 41.0], atol=1e-14)
assert np.isclose(rec_split["chi2_stat"], (3.0 ** 2) / 6.0, rtol=1e-14)
print("Measured flows   [kg/h]:", y_meas_split, f"| Raw imbalance = {rec_split['balance_residual_before'][0]:+.2f} kg/h")
print("Reconciled flows [kg/h]:", rec_split["y_reconciled"], f"| New imbalance = {rec_split['balance_residual_after'][0]:+.1e} kg/h")
print(f"Adjustments      [kg/h]: {rec_split['y_reconciled'] - y_meas_split} (Proportional to variances [1, 1, 4])")
print(f"Global chi^2 statistic : {rec_split['chi2_stat']:.4f} (df = {rec_split['dof']})")
heading("Demonstration 2: 5-Stream Reactor-Separator-Recycle Incidence & Residual Covariance")
A_net = np.array([
    [1.0, -1.0,  0.0,  0.0,  1.0],  # Node 1: Mixer
    [0.0,  1.0, -1.0,  0.0,  0.0],  # Node 2: Reactor
    [0.0,  0.0,  1.0, -1.0, -1.0],  # Node 3: Separator
])
b_net = np.zeros(3)
y_true_net = np.array([100.0, 140.0, 140.0, 100.0, 40.0])
sigma_net = np.array([0.5, 1.2, 1.2, 0.5, 2.0])

rng = np.random.default_rng(101)
y_normal = y_true_net + rng.normal(0.0, sigma_net)
rec_normal = linear_physics.crowe_data_reconciliation(y_normal, sigma_net, A_net, b_net)
print("Raw nodal balance residuals [Mixer, Reactor, Separator]:", np.round(rec_normal["balance_residual_before"], 4))
print("Post-reconciliation nodal residuals                    :", np.round(rec_normal["balance_residual_after"], 15))
print(f"Normal operation chi^2 = {rec_normal['chi2_stat']:.3f} < chi^2_(3, 0.95) = 7.815 (No gross error alarm)")
assert np.max(np.abs(rec_normal["balance_residual_after"])) < 1e-13 and rec_normal["chi2_stat"] < 7.815
heading("Demonstration 3: Gross-Error Injection & Faulty Sensor Isolation via Measurement z-Test")
y_faulty = y_normal.copy()
y_faulty[2] += 12.0  # Inject +12 kg/h gross bias into Stream 3 (Reactor Outlet flowmeter)
rec_faulty = linear_physics.crowe_data_reconciliation(y_faulty, sigma_net, A_net, b_net)
fault_sensor_idx = int(np.argmax(rec_faulty["measurement_z"]))

print(f"Faulty operation chi^2 = {rec_faulty['chi2_stat']:.2f} >> 7.815 (gross error alarm triggered)")
print(f"{'Stream':>8s} | {'True':>7s} | {'Measured':>9s} | {'Reconciled':>10s} | {'Meas z-score':>12s}")
print("-" * 58)
for i in range(5):
    flag = " <-- largest!" if i == fault_sensor_idx else ""
    print(f"Stream {i+1:1d} | {y_true_net[i]:7.2f} | {y_faulty[i]:9.2f} | {rec_faulty['y_reconciled'][i]:10.2f} | {rec_faulty['measurement_z'][i]:12.2f}{flag}")
assert rec_faulty["chi2_stat"] > 30.0 and fault_sensor_idx == 2
heading("Demonstration 4: Gross-Error Magnitude Sweep & Chi-Squared Power Curve")
print(f"{'Bias dY_3 [kg/h]':>16s} | {'Global chi^2':>12s} | {'Max z_meas':>10s} | {'Isolated Stream':>15s} | {'Alarm Status':>14s}")
print("-" * 76)
for bias in [0.0, 2.0, 4.5, 7.0, 12.0]:
    y_sw = y_normal.copy()
    y_sw[2] += bias
    r_sw = linear_physics.crowe_data_reconciliation(y_sw, sigma_net, A_net, b_net)
    idx_max = int(np.argmax(r_sw["measurement_z"])) + 1
    status = "ALARM (>99%)" if r_sw["chi2_stat"] > 11.345 else ("WARN (>95%)" if r_sw["chi2_stat"] > 7.815 else "Normal")
    print(f"{bias:16.1f} | {r_sw['chi2_stat']:12.2f} | {np.max(r_sw['measurement_z']):10.2f} | {'Stream ' + str(idx_max):>15s} | {status:>14s}")
heading("Inside the Algorithm: By-Hand Crowe Projection vs. the Library")
Sigma_mat = np.diag(sigma_net ** 2)
r_vec = A_net @ y_normal - b_net
S_r_mat = A_net @ Sigma_mat @ A_net.T
y_rec_hand = y_normal - Sigma_mat @ A_net.T @ np.linalg.solve(S_r_mat, r_vec)
diff_crowe = float(np.max(np.abs(y_rec_hand - rec_normal["y_reconciled"])))
print("by hand (pure NumPy):", np.round(y_rec_hand, 6))
print("library (engml_2_applied)     :", np.round(rec_normal["y_reconciled"], 6))
print(f"Max discrepancy     : {diff_crowe:.2e}")
assert diff_crowe < 1e-13
sys.stdout = _stdout
print("Set-up complete: the variables of the lecture's demonstrations are defined.")

Set-up complete: the variables of the lecture's demonstrations are defined.


## Exercise 1

**Problem.** In the 5-stream reactor-separator network (`A_net`, `b_net`, `sigma_net`), inject a $-10.0\;\mathrm{kg/h}$ gross bias into the recycle orifice meter (Stream 5, index `4`) on top of `y_true_net`. Run `crowe_data_reconciliation`, verify that $\chi^2_{\mathrm{stat}} > 15.0$, and confirm that `np.argmax(rec['measurement_z']) == 4` isolates Stream 5.

In [2]:
heading("Exercise 1: Recycle Orifice Meter Gross-Error Isolation")
y_bias5 = y_true_net.copy()
y_bias5[4] -= 10.0
rec_bias5 = linear_physics.crowe_data_reconciliation(y_bias5, sigma_net, A_net, b_net)
isolated_idx = int(np.argmax(rec_bias5["measurement_z"]))
print(f"Recycle fault chi^2 = {rec_bias5['chi2_stat']:.2f} | Isolated stream index = {isolated_idx} (z = {rec_bias5['measurement_z'][isolated_idx]:.2f})")
assert rec_bias5["chi2_stat"] > 15.0 and isolated_idx == 4
assert np.max(np.abs(rec_bias5["balance_residual_after"])) < 1e-13


Exercise 1: Recycle Orifice Meter Gross-Error Isolation
Recycle fault chi^2 = 20.64 | Isolated stream index = 4 (z = 4.54)


**Discussion.** Even though the recycle orifice meter has the largest uncertainty ($\sigma_5 = 2.0\;\mathrm{kg/h}$), a $-10.0\;\mathrm{kg/h}$ ($5\sigma_5$) bias violates the Mixer and Separator balances simultaneously, which gives $\chi^2 = 20.6$ and the largest measurement statistic at index `4` ($z = 4.54$).

---

## Exercise 2

**Problem.** Quantify the smearing damage caused by reconciling a flowsheet *without* removing a faulty sensor first: compare the mean absolute error (MAE) across the 4 healthy streams (`[0, 1, 3, 4]`) when `y_faulty` (with the $+12\;\mathrm{kg/h}$ bias on Stream 3) is reconciled directly, versus replacing the faulty Stream 3 uncertainty with a huge dummy standard deviation $\sigma_3 = 1000\;\mathrm{kg/h}$ (effectively unweighting the broken meter).

In [3]:
heading("Exercise 2: Smearing Damage Audit & Faulty-Sensor Downweighting")
healthy_mask = [0, 1, 3, 4]
mae_smeared = float(np.mean(np.abs(rec_faulty["y_reconciled"][healthy_mask] - y_true_net[healthy_mask])))
sigma_cleaned = sigma_net.copy()
sigma_cleaned[2] = 1000.0  # Unweight isolated faulty sensor #3
rec_cleaned = linear_physics.crowe_data_reconciliation(y_faulty, sigma_cleaned, A_net, b_net)
mae_cleaned = float(np.mean(np.abs(rec_cleaned["y_reconciled"][healthy_mask] - y_true_net[healthy_mask])))
print(f"Healthy-stream MAE with unremoved fault (smearing) : {mae_smeared:.3f} kg/h")
print(f"Healthy-stream MAE after downweighting Stream 3    : {mae_cleaned:.3f} kg/h")
print(f"Reconstructed flow for broken Stream 3             : {rec_cleaned['y_reconciled'][2]:.2f} kg/h (True = 140.00)")
assert mae_cleaned < 0.55 * mae_smeared and rec_cleaned["chi2_stat"] < 2.0


Exercise 2: Smearing Damage Audit & Faulty-Sensor Downweighting
Healthy-stream MAE with unremoved fault (smearing) : 2.176 kg/h
Healthy-stream MAE after downweighting Stream 3    : 1.000 kg/h
Reconstructed flow for broken Stream 3             : 138.03 kg/h (True = 140.00)


**Discussion.** Reconciling with the faulty meter left in spreads its error over the others: their mean absolute error is $2.18\;\mathrm{kg/h}$. With the faulty meter given a very large variance, the mean absolute error of the healthy meters is $1.00\;\mathrm{kg/h}$, and the balances reconstruct the flow of Stream 3 as $138.0\;\mathrm{kg/h}$ (true value $140$).

---

## Exercise 3

**Problem.** Implement an automated gross-error elimination function `iterative_crowe_reconcile(y_meas, sigma_meas, A, b, chi2_threshold=7.815)` that checks $\chi^2_{\mathrm{stat}} > \chi^2_{\mathrm{threshold}}$, identifies the sensor with maximum `measurement_z`, inflates its standard deviation by $10^4\times$ to remove its corrupting leverage, and re-runs Crowe reconciliation. Verify it on `y_faulty`.

In [4]:
heading("Exercise 3: Implement Automated Gross-Error Elimination & Reconciliation")
def iterative_crowe_reconcile(
    y_meas: np.ndarray, sigma_meas: np.ndarray, A: np.ndarray, b: np.ndarray, chi2_threshold: float = 7.815
) -> dict:
    """Detect gross error via chi^2 test, unweight the highest-z sensor, and reconcile."""
    sig_curr = np.asarray(sigma_meas, dtype=float).copy()
    rec = linear_physics.crowe_data_reconciliation(y_meas, sig_curr, A, b)
    flagged = []
    if rec["chi2_stat"] > chi2_threshold:
        bad_idx = int(np.argmax(rec["measurement_z"]))
        flagged.append(bad_idx)
        sig_curr[bad_idx] *= 1e4
        rec = linear_physics.crowe_data_reconciliation(y_meas, sig_curr, A, b)
    rec["flagged_sensors"] = flagged
    return rec

auto_res = iterative_crowe_reconcile(y_faulty, sigma_net, A_net, b_net)
print("Flagged sensor indices:", auto_res["flagged_sensors"])
print("Cleaned reconciled flows [kg/h]:", np.round(auto_res["y_reconciled"], 2))
print(f"Post-cleaning chi^2 = {auto_res['chi2_stat']:.4f} | Max balance residual = {np.max(np.abs(auto_res['balance_residual_after'])):.2e}")
assert auto_res["flagged_sensors"] == [2] and auto_res["chi2_stat"] < 2.0


Exercise 3: Implement Automated Gross-Error Elimination & Reconciliation
Flagged sensor indices: [2]
Cleaned reconciled flows [kg/h]: [ 99.95 138.03 138.03  99.95  38.08]
Post-cleaning chi^2 = 1.7664 | Max balance residual = 4.98e-10


**Discussion.** Reconciliation followed by the $\chi^2$ test and the removal of the measurement with the largest standardized adjustment keeps one biased instrument from spreading its error over the other streams.